# Temporal DVC showcase — `MultiGPUDispatcher` + `correlate_series`

Self-contained walkthrough of the time-series pipeline that landed on the
`temporal-buildout` branch:

1. Build a textured reference volume (band-limited noise).
2. Define a time-parameterized analytical displacement field
   (`rigid_shift` + `uniform_dilation`, scaled linearly in time).
3. Synthesize the warped frame series via `make_series`.
4. Visualize the reference + the analytical field at sample timesteps.
5. Configure DVC inputs in one cell.
6. Run **`SEQUENTIAL`** and **`REFERENCE_ANCHORED`** strategies through a
   single persistent `MultiGPUDispatcher` (paid once per series).
7. Quantify error vs. ground truth: per-pair MAE/RMSE/p95 plus cumulative
   drift for `SEQUENTIAL`, with sample-POI traces and side-by-side
   recovered-vs-truth slices.

The notebook degrades gracefully: if CuPy is not available it falls back to
the host-only `correlate()` path through `correlate_series(dispatcher=None)`.

**Sign convention.** Frames are synthesized with the pull-back convention
(`deformed(x) = reference(x - u(x))`), which is what `correlate()` recovers
directly — no sign flips when comparing recovered to analytical.

In [ ]:
from __future__ import annotations

import time
import warnings
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np

from mamba_dvc.core.grid import build_grid
from mamba_dvc.pipeline.series import correlate_series
from mamba_dvc.types import (
    DisplacementField,
    DisplacementSeries,
    PairingStrategy,
    POIStatus,
    SeriesPairStatus,
)
from mamba_dvc.validate.synthetic import (
    compose,
    linear_motion,
    make_series,
    make_texture,
    rigid_shift,
    uniform_dilation,
)

try:
    import cupy as _cp  # noqa: F401
    from mamba_dvc.gpu.dispatch import MultiGPUDispatcher
    _HAS_CUPY = True
except ImportError:
    MultiGPUDispatcher = None  # type: ignore[assignment]
    _HAS_CUPY = False

print(f"CuPy available: {_HAS_CUPY}")

## 1. Setup — DVC algorithm + experiment parameters

Single place to tune the run. The volume side defaults to `192` so a full
16-frame, two-strategy sweep finishes in a couple of minutes on one A6000
and in 5–10 min on CPU. For full-resolution `(960, 1280, 1280)` runs raise
the window to `96` (the production default in `docs/plans/overview.md`).

In [ ]:
# --- Volume / series ----------------------------------------------------
VOLUME_SHAPE: tuple[int, int, int] = (192, 192, 192)
N_TIMESTEPS = 16                       # frames at t = 0, 1, ..., N-1
TEXTURE_SIGMA = 1.5                    # vx — low-pass for the noise texture
TEXTURE_SEED = 0
WARP_ORDER = 3                         # cubic spline; 1 inflates the error floor

# --- Time-parameterized displacement field ------------------------------
PER_STEP_SHIFT = (0.5, 0.8, 0.2)       # (dz, dy, dx) voxels added per t step
PER_STEP_STRAIN = 5e-4                 # uniform dilation strain per t step

# --- DVC correlator (forwarded straight to correlate_series) ------------
WINDOW = 64                            # subvolume size; fits ~125 POIs in 192**3
OVERLAP = 0.5                          # window overlap fraction
MASK_THRESHOLD = 0.9                   # POI admission threshold against `mask`
SEARCH_RADIUS = None                   # None -> window // 2
TUKEY_ALPHA = None                     # None -> mode-dependent default
BATCH_SIZE: int | str = "auto"        # VRAM-aware oracle when GPUs are visible
EPS = 1e-12

# --- Dispatcher --------------------------------------------------------
# None  -> auto-discover every visible CUDA device (production behaviour).
# []    -> force CPU path even if GPUs are available.
# [0]   -> single-GPU in-process (no spawn).
# [0,1] -> multi-process pool across the listed devices.
USE_DISPATCHER = _HAS_CUPY
DEVICE_IDS: list[int] | None = None

print("Volume shape   :", VOLUME_SHAPE)
print("Timesteps      :", N_TIMESTEPS, "->", tuple(range(N_TIMESTEPS)))
print("Per-step shift :", PER_STEP_SHIFT, "vx/step")
print("Per-step strain:", PER_STEP_STRAIN)
print("Window/overlap :", WINDOW, "/", OVERLAP)
print("Batch size     :", BATCH_SIZE)
print("Use dispatcher :", USE_DISPATCHER, "  devices:", DEVICE_IDS)

## 2. Reference volume

`make_texture` returns a Gaussian-low-passed white-noise volume,
standardized to zero mean / unit variance. The `sigma=1.5` low-pass keeps
energy across the band the FFT correlator uses while remaining smooth
enough that cubic resampling stays faithful — see
`mamba_dvc/validate/synthetic.py` for the rationale.

In [ ]:
reference = make_texture(VOLUME_SHAPE, sigma=TEXTURE_SIGMA, seed=TEXTURE_SEED)
print(
    f"reference  shape={reference.shape}  dtype={reference.dtype}  "
    f"mean={reference.mean():+.3f}  std={reference.std():.3f}"
)

In [ ]:
def show_orthoslices(volume: np.ndarray, title: str) -> None:
    """Mid-plane slices along z / y / x with a shared colormap."""
    z_mid, y_mid, x_mid = (s // 2 for s in volume.shape)
    vmin, vmax = np.percentile(volume, [1, 99])
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    for ax, plane, label in zip(
        axes,
        [volume[z_mid], volume[:, y_mid], volume[:, :, x_mid]],
        [f"z={z_mid}", f"y={y_mid}", f"x={x_mid}"],
        strict=True,
    ):
        im = ax.imshow(plane, cmap="gray", vmin=vmin, vmax=vmax)
        ax.set_title(label)
        ax.set_axis_off()
    fig.colorbar(im, ax=axes, shrink=0.7, pad=0.02)
    fig.suptitle(title)
    plt.show()

show_orthoslices(reference, "Reference texture (mid-plane orthoslices)")

## 3. Time-parameterized deformation field

Compose a per-step rigid shift with a small uniform dilation about the
volume centroid, then wrap with `linear_motion` so the field scales with
time as `u(coords, t) = t * velocity(coords)`. `make_series` materializes
one warped frame per timestep against the same reference.

In [ ]:
timesteps = tuple(range(N_TIMESTEPS))
center = tuple(s / 2.0 for s in VOLUME_SHAPE)

velocity = compose(
    rigid_shift(PER_STEP_SHIFT),
    uniform_dilation(PER_STEP_STRAIN, center),
)
u_of_t = linear_motion(velocity)

t0 = time.perf_counter()
series = make_series(
    VOLUME_SHAPE,
    u_of_t,
    timesteps,
    reference=reference,
    order=WARP_ORDER,
    convention="pull_back",
)
print(
    f"Synthesized {len(series.frames)} frames in {time.perf_counter() - t0:.1f} s"
)
print(
    f"  frame[0]  std={series.frames[0].std():.3f}  "
    f"frame[-1] std={series.frames[-1].std():.3f}"
)

### 3a. Visualize the analytical field magnitude over time

Sample `u(coords, t)` on a coarse lattice and plot the per-axis component
and the Euclidean magnitude at the mid-z slice for a few timesteps.

In [ ]:
def sample_field_slice(
    field_at: object,
    t: float,
    shape: tuple[int, int, int],
    z_idx: int,
    stride: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """Evaluate the temporal field at one z slice and return (coords, disp)."""
    yy, xx = np.meshgrid(
        np.arange(0, shape[1], stride, dtype=np.float32),
        np.arange(0, shape[2], stride, dtype=np.float32),
        indexing="ij",
    )
    coords = np.stack(
        [np.full(yy.size, z_idx, dtype=np.float32), yy.ravel(), xx.ravel()],
        axis=1,
    )
    disp = field_at(coords, t)
    return coords, disp


z_mid = VOLUME_SHAPE[0] // 2
preview_ts = [timesteps[1], timesteps[len(timesteps) // 2], timesteps[-1]]
stride = 6

fig, axes = plt.subplots(1, len(preview_ts), figsize=(4 * len(preview_ts), 4))
for ax, t in zip(axes, preview_ts, strict=True):
    coords, disp = sample_field_slice(series.field_at, float(t), VOLUME_SHAPE, z_mid, stride)
    mag = np.linalg.norm(disp, axis=1).reshape(
        VOLUME_SHAPE[1] // stride, VOLUME_SHAPE[2] // stride
    )
    im = ax.imshow(mag, origin="lower", cmap="viridis")
    # Subsample arrows for legibility.
    arrow_stride = 3
    yi = coords[:, 1].reshape(mag.shape) / stride
    xi = coords[:, 2].reshape(mag.shape) / stride
    dy = disp[:, 1].reshape(mag.shape)
    dx = disp[:, 2].reshape(mag.shape)
    ax.quiver(
        xi[::arrow_stride, ::arrow_stride],
        yi[::arrow_stride, ::arrow_stride],
        dx[::arrow_stride, ::arrow_stride],
        dy[::arrow_stride, ::arrow_stride],
        color="white",
        scale=80.0,
        width=0.005,
    )
    ax.set_title(f"t={t}  ||u|| at z={z_mid}")
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.75, label="voxels")
plt.tight_layout()
plt.show()

In [ ]:
# Show that frame[-1] really moved relative to frame[0].
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, vol, title in zip(
    axes,
    [series.frames[0], series.frames[-1], series.frames[-1] - series.frames[0]],
    [f"frame[t={timesteps[0]}]", f"frame[t={timesteps[-1]}]", "difference"],
    strict=True,
):
    plane = vol[z_mid]
    vmin, vmax = np.percentile(plane, [1, 99])
    if title == "difference":
        m = float(np.max(np.abs(plane)))
        im = ax.imshow(plane, cmap="RdBu_r", vmin=-m, vmax=m)
    else:
        im = ax.imshow(plane, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.75)
fig.suptitle(f"Mid-z slice (z={z_mid})")
plt.tight_layout()
plt.show()

## 4. Run the DVC pipeline — both strategies through one dispatcher

The persistent `MultiGPUDispatcher` pays the spawn + CUDA-init cost once on
`__enter__`; each `correlate_series` call thereafter is compute-only. We
reuse the same opened dispatcher across both strategies in this session.

When `USE_DISPATCHER=False` (CPU-only host) `correlate_series` falls back
to the pure `correlate()` path. The notebook code path is identical.

In [ ]:
def open_dispatcher():
    if USE_DISPATCHER and MultiGPUDispatcher is not None:
        return MultiGPUDispatcher(
            device_ids=DEVICE_IDS,
            volume_shape=VOLUME_SHAPE,
            window=WINDOW,
            overlap=OVERLAP,
            mask_threshold=MASK_THRESHOLD,
            tukey_alpha=TUKEY_ALPHA,
            search_radius=SEARCH_RADIUS,
            batch_size=BATCH_SIZE,
            eps=EPS,
        )
    return nullcontext(None)


def run_strategy(
    strategy: PairingStrategy,
    dispatcher: MultiGPUDispatcher | None,
) -> DisplacementSeries:
    """Drive the frame iterator through correlate_series for one strategy."""
    frames_iter = ((t, frame) for t, frame in zip(series.timesteps, series.frames, strict=True))
    return correlate_series(
        frames_iter,
        mask=None,
        strategy=strategy,
        dispatcher=dispatcher,
        window=WINDOW,
        overlap=OVERLAP,
        mask_threshold=MASK_THRESHOLD,
        tukey_alpha=TUKEY_ALPHA,
        search_radius=SEARCH_RADIUS,
        batch_size=BATCH_SIZE,
        eps=EPS,
    )


results: dict[PairingStrategy, DisplacementSeries] = {}
timings: dict[PairingStrategy, float] = {}

with open_dispatcher() as dispatcher:
    if dispatcher is not None:
        print(f"Dispatcher opened on devices {dispatcher.device_ids}")
        print(f"  multiprocess: {dispatcher.is_multiprocess}")
    else:
        print("Dispatcher: none (host-only correlate fallback)")

    for strategy in (PairingStrategy.SEQUENTIAL, PairingStrategy.REFERENCE_ANCHORED):
        with warnings.catch_warnings():
            warnings.simplefilter("always", RuntimeWarning)
            t0 = time.perf_counter()
            results[strategy] = run_strategy(strategy, dispatcher)
            timings[strategy] = time.perf_counter() - t0
        n_ok = int(np.sum(results[strategy].pair_status == int(SeriesPairStatus.OK)))
        n_total = len(results[strategy].fields)
        print(
            f"  {strategy.value:20s} pairs={n_total:3d}  ok={n_ok:3d}  "
            f"elapsed={timings[strategy]:.1f} s"
        )

## 5. Per-pair error vs. analytical ground truth

For each pair `(t_ref, t_def)`, compare the recovered displacement at every
valid POI to the analytical inter-frame field
`u(x, t_def) - u(x, t_ref)`. The shared grid lives on the series so GT
sampling and DVC POIs are colocated by construction.

In [ ]:
def per_pair_error(series_result: DisplacementSeries) -> dict[str, np.ndarray]:
    """Return per-pair MAE / RMSE / p95 / n_valid arrays vs. analytical GT."""
    grid = series_result.grid
    n_pairs = len(series_result.fields)
    mae = np.full(n_pairs, np.nan, dtype=np.float32)
    rmse = np.full(n_pairs, np.nan, dtype=np.float32)
    p95 = np.full(n_pairs, np.nan, dtype=np.float32)
    n_valid = np.zeros(n_pairs, dtype=np.int64)

    for i, (field, (t_ref, t_def), status) in enumerate(
        zip(
            series_result.fields,
            series_result.pair_indices,
            series_result.pair_status,
            strict=True,
        )
    ):
        if int(status) != int(SeriesPairStatus.OK):
            continue
        gt = series.field_at(grid.positions, float(t_def)) - series.field_at(
            grid.positions, float(t_ref)
        )
        err = field.displacements - gt
        valid = field.valid
        if not bool(valid.any()):
            continue
        mag = np.linalg.norm(err[valid], axis=1)
        mae[i] = float(mag.mean())
        rmse[i] = float(np.sqrt((mag**2).mean()))
        p95[i] = float(np.percentile(mag, 95))
        n_valid[i] = int(valid.sum())
    return {"mae": mae, "rmse": rmse, "p95": p95, "n_valid": n_valid}


errors = {strategy: per_pair_error(s) for strategy, s in results.items()}

for strategy, stats in errors.items():
    finite = np.isfinite(stats["mae"])
    print(
        f"{strategy.value:20s}  pairs={int(finite.sum())}  "
        f"MAE med={np.median(stats['mae'][finite]):.4f}  "
        f"RMSE med={np.median(stats['rmse'][finite]):.4f}  "
        f"p95 med={np.median(stats['p95'][finite]):.4f} vx"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True)
metrics = [("mae", "MAE"), ("rmse", "RMSE"), ("p95", "p95")]
colors = {PairingStrategy.SEQUENTIAL: "tab:blue", PairingStrategy.REFERENCE_ANCHORED: "tab:orange"}

for ax, (key, label) in zip(axes, metrics, strict=True):
    for strategy, stats in errors.items():
        t_def = results[strategy].pair_indices[:, 1]
        ax.plot(
            t_def,
            stats[key],
            marker="o",
            ms=3,
            color=colors[strategy],
            label=strategy.value,
        )
    ax.set_xlabel("t_def (deformed-frame index)")
    ax.set_ylabel(f"{label} [voxels]")
    ax.set_title(f"Per-pair {label}")
    ax.grid(alpha=0.3)
axes[0].legend(loc="best")
plt.tight_layout()
plt.show()

**Reading the plot.** `REFERENCE_ANCHORED` errors grow with `t` because the
inter-frame displacement to recover grows linearly. `SEQUENTIAL` errors
stay flat per pair because every pair has only one step of motion to
resolve — but those errors random-walk when composed, which §6 unpacks.

### 5a. Sample POIs — recovered vs. analytical displacement

Pick a handful of POIs from the center of the grid and trace their
recovered per-axis displacements against the analytical GT across the
series. For `REFERENCE_ANCHORED` this is the absolute displacement at
`t`; for `SEQUENTIAL` it is the per-step increment, which the analytical
field provides as `u(x, t) - u(x, t-1)`.

In [ ]:
rng = np.random.default_rng(7)
grid_for_samples = results[PairingStrategy.SEQUENTIAL].grid
n_pois = grid_for_samples.positions.shape[0]
n_samples = 4
sample_idx = rng.choice(n_pois, size=n_samples, replace=False)
sample_idx.sort()
sample_positions = grid_for_samples.positions[sample_idx]
print("Sampled POIs (z, y, x):")
for i, p in zip(sample_idx, sample_positions, strict=True):
    print(f"  idx={int(i):4d}  pos=({p[0]:6.1f}, {p[1]:6.1f}, {p[2]:6.1f})")

In [ ]:
def gather_traces(
    series_result: DisplacementSeries,
    sample_idx: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (t_def, recovered, gt) with shape (pairs, samples, 3)."""
    pairs = series_result.pair_indices
    n_pairs = pairs.shape[0]
    n_samples = sample_idx.shape[0]
    rec = np.full((n_pairs, n_samples, 3), np.nan, dtype=np.float32)
    gt = np.full((n_pairs, n_samples, 3), np.nan, dtype=np.float32)
    grid = series_result.grid
    coords = grid.positions[sample_idx]
    for i, (field, (t_ref, t_def), status) in enumerate(
        zip(
            series_result.fields,
            pairs,
            series_result.pair_status,
            strict=True,
        )
    ):
        if int(status) != int(SeriesPairStatus.OK):
            continue
        rec[i] = field.displacements[sample_idx]
        gt[i] = series.field_at(coords, float(t_def)) - series.field_at(coords, float(t_ref))
    return pairs[:, 1], rec, gt


traces = {s: gather_traces(results[s], sample_idx) for s in results}

In [ ]:
axis_labels = ["dz", "dy", "dx"]

for strategy, (t_def, rec, gt) in traces.items():
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
    for axis, ax in enumerate(axes):
        for s_idx, (idx, pos) in enumerate(zip(sample_idx, sample_positions, strict=True)):
            color = f"C{s_idx}"
            ax.plot(t_def, gt[:, s_idx, axis], color=color, linestyle="--", alpha=0.6)
            ax.plot(
                t_def,
                rec[:, s_idx, axis],
                color=color,
                marker="o",
                ms=3,
                label=f"POI {int(idx)}  ({pos[0]:.0f},{pos[1]:.0f},{pos[2]:.0f})",
            )
        ax.set_title(f"{axis_labels[axis]} component")
        ax.set_xlabel("t_def")
        ax.set_ylabel("displacement [voxels]")
        ax.grid(alpha=0.3)
    axes[0].legend(loc="best", fontsize=8)
    fig.suptitle(f"{strategy.value}: solid = recovered, dashed = analytical")
    plt.tight_layout()
    plt.show()

## 6. Cumulative drift — `SEQUENTIAL` composed vs. analytical absolute

`DisplacementSeries.cumulative()` chains the per-step `SEQUENTIAL` fields
via semi-Lagrangian composition. Compared against the analytical
*absolute* displacement `u(x, t) - u(x, t_start)`, this captures the
random-walk drift that the per-pair plot in §5 hides.

For `REFERENCE_ANCHORED`, `cumulative()` returns the OK-prefix verbatim —
the per-pair plot is already the drift plot — so we only report drift for
`SEQUENTIAL` and compare its absolute error against the anchored result
side-by-side.

In [ ]:
def absolute_error(
    composed: tuple[DisplacementField, ...],
    t_start: int,
    t_steps: np.ndarray,
) -> dict[str, np.ndarray]:
    """Stats of composed (or anchored) fields vs. absolute GT u(x,t) - u(x,t_start)."""
    if not composed:
        return {
            "t": np.empty(0, dtype=np.int64),
            "mae": np.empty(0, dtype=np.float32),
            "rmse": np.empty(0, dtype=np.float32),
            "p95": np.empty(0, dtype=np.float32),
            "n_valid": np.empty(0, dtype=np.int64),
        }
    n = len(composed)
    mae = np.full(n, np.nan, dtype=np.float32)
    rmse = np.full(n, np.nan, dtype=np.float32)
    p95 = np.full(n, np.nan, dtype=np.float32)
    n_valid = np.zeros(n, dtype=np.int64)
    for i, field in enumerate(composed):
        gt_abs = series.field_at(field.positions, float(t_steps[i])) - series.field_at(
            field.positions, float(t_start)
        )
        err = field.displacements - gt_abs
        valid = field.valid
        if not bool(valid.any()):
            continue
        mag = np.linalg.norm(err[valid], axis=1)
        mae[i] = float(mag.mean())
        rmse[i] = float(np.sqrt((mag**2).mean()))
        p95[i] = float(np.percentile(mag, 95))
        n_valid[i] = int(valid.sum())
    return {"t": np.asarray(t_steps[:n], dtype=np.int64), "mae": mae, "rmse": rmse, "p95": p95, "n_valid": n_valid}


seq_result = results[PairingStrategy.SEQUENTIAL]
anchored_result = results[PairingStrategy.REFERENCE_ANCHORED]

composed_seq = seq_result.cumulative(interpolation="linear")
t_start_seq = int(seq_result.pair_indices[0, 0]) if len(seq_result.pair_indices) else 0
t_steps_seq = seq_result.pair_indices[: len(composed_seq), 1]

anchored_t_start = int(anchored_result.pair_indices[0, 0]) if len(anchored_result.pair_indices) else 0
anchored_t_steps = anchored_result.pair_indices[:, 1]

seq_drift = absolute_error(composed_seq, t_start_seq, t_steps_seq)
anchored_drift = absolute_error(anchored_result.fields, anchored_t_start, anchored_t_steps)

print(
    f"SEQUENTIAL  composed steps : {len(composed_seq)}/{len(seq_result.fields)}"
)
if len(seq_drift["mae"]):
    print(
        f"  drift MAE  first={seq_drift['mae'][0]:.4f}  last={seq_drift['mae'][-1]:.4f}"
    )
if len(anchored_drift["mae"]):
    print(
        f"  anchored MAE first={anchored_drift['mae'][0]:.4f}  last={anchored_drift['mae'][-1]:.4f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, key, title in zip(axes, ["mae", "p95"], ["absolute MAE", "absolute p95"], strict=True):
    ax.plot(
        seq_drift["t"],
        seq_drift[key],
        marker="o",
        ms=4,
        color="tab:blue",
        label="sequential (composed)",
    )
    ax.plot(
        anchored_drift["t"],
        anchored_drift[key],
        marker="s",
        ms=4,
        color="tab:orange",
        label="reference_anchored",
    )
    ax.set_xlabel("t_def")
    ax.set_ylabel(f"{title} [voxels]")
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].legend(loc="best")
fig.suptitle("Absolute displacement error vs t  (composed SEQUENTIAL vs anchored)")
plt.tight_layout()
plt.show()

## 7. Spatial sanity — recovered vs. GT magnitude on a mid-grid slice

Reshape the per-POI fields back to lattice form and compare a mid-z
slice of recovered displacement magnitude against the analytical magnitude
for the last `REFERENCE_ANCHORED` pair (where the displacement is largest
and the picture is most informative).

In [ ]:
last_anchored = anchored_result.fields[-1]
grid_shape = last_anchored.grid_shape
disp_lattice = last_anchored.displacements.reshape(*grid_shape, 3)
valid_lattice = last_anchored.valid.reshape(grid_shape)

t_last = int(anchored_result.pair_indices[-1, 1])
gt_at_grid = series.field_at(
    last_anchored.positions, float(t_last)
) - series.field_at(last_anchored.positions, float(anchored_t_start))
gt_lattice = gt_at_grid.reshape(*grid_shape, 3)

rec_mag = np.linalg.norm(disp_lattice, axis=-1)
gt_mag = np.linalg.norm(gt_lattice, axis=-1)
err_mag = np.linalg.norm(disp_lattice - gt_lattice, axis=-1)
rec_mag_masked = np.where(valid_lattice, rec_mag, np.nan)
err_mag_masked = np.where(valid_lattice, err_mag, np.nan)

z_lat_mid = grid_shape[0] // 2
vmax_field = float(np.nanmax([rec_mag_masked[z_lat_mid], gt_mag[z_lat_mid]]))
vmax_err = float(np.nanmax(err_mag_masked[z_lat_mid]))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = [
    (rec_mag_masked[z_lat_mid], f"recovered  ||u||  t={t_last}", "viridis", 0.0, vmax_field),
    (gt_mag[z_lat_mid], f"analytical ||u||  t={t_last}", "viridis", 0.0, vmax_field),
    (err_mag_masked[z_lat_mid], "|recovered − analytical|", "magma", 0.0, vmax_err),
]
for ax, (plane, title, cmap, vmin, vmax) in zip(axes, panels, strict=True):
    im = ax.imshow(plane, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, shrink=0.8, label="voxels")
fig.suptitle(f"REFERENCE_ANCHORED last pair — mid-z grid slice (z_lat={z_lat_mid})")
plt.tight_layout()
plt.show()

## 8. Summary table

Aggregate per-pair and (where applicable) cumulative-drift medians for the
two strategies plus the wall-clock cost. With a single 192³ texture and
16 frames the absolute error floor on textured noise sits in the
`0.01–0.05 vx` range — anything materially above that is a regression
candidate.

In [ ]:
def median_finite(arr: np.ndarray) -> float:
    finite = arr[np.isfinite(arr)]
    return float(np.median(finite)) if finite.size else float("nan")


rows = []
for strategy, e in errors.items():
    rows.append(
        (
            strategy.value,
            "per-pair",
            median_finite(e["mae"]),
            median_finite(e["rmse"]),
            median_finite(e["p95"]),
            timings[strategy],
        )
    )
if len(seq_drift["mae"]):
    rows.append(
        (
            "sequential (composed)",
            "cumulative",
            median_finite(seq_drift["mae"]),
            median_finite(seq_drift["rmse"]),
            median_finite(seq_drift["p95"]),
            float("nan"),
        )
    )
if len(anchored_drift["mae"]):
    rows.append(
        (
            "reference_anchored (absolute)",
            "cumulative",
            median_finite(anchored_drift["mae"]),
            median_finite(anchored_drift["rmse"]),
            median_finite(anchored_drift["p95"]),
            float("nan"),
        )
    )

header = f"{'strategy':30s}  {'kind':10s}  {'MAE':>7s}  {'RMSE':>7s}  {'p95':>7s}  {'elapsed':>8s}"
print(header)
print("-" * len(header))
for name, kind, mae, rmse, p95, elapsed in rows:
    elapsed_str = f"{elapsed:7.1f}s" if np.isfinite(elapsed) else "     n/a"
    print(f"{name:30s}  {kind:10s}  {mae:7.4f}  {rmse:7.4f}  {p95:7.4f}  {elapsed_str}")

### Next steps

- **Anchored fast-path.** For a `REFERENCE_ANCHORED`-only series, build
  the dispatcher with `anchored_reference=reference` so `frame_0` is
  uploaded to every GPU once and the driver skips re-uploading it per
  pair. The notebook above mixes strategies, so it deliberately does not
  bind an anchored reference (see `MultiGPUDispatcher.__init__` docstring).
- **Auto-batch logging.** When `BATCH_SIZE="auto"`, the dispatcher emits a
  `auto.batch_size=… auto.free_bytes_per_device=…` line on stderr at
  `__enter__`. Capture it if you need to log the resolved batch per run.
- **Per-frame masks.** v1 reuses one mask across the series; per-frame
  deformed masks are reserved for v2.
- **Sweeping lags / amplitudes.** For a multi-config sweep prefer
  `mamba_dvc.validate.series_error.evaluate_synthetic`, which threads a
  single dispatcher through the full grid (see
  `scripts/eval_temporal_strategies.py --devices 0,1,2,3`).

## §9 Debug ladder — small-displacement error

Adapted from `notebooks/dvc-debug-texture.ipynb`. The error pattern we observed
(MAE 6+ vx at `t=1`, 0.05 vx at `t=15`) is inverted relative to the
`|u|/W` shrinkage bug that the prior debug session diagnosed, so the prior
fix is not regressed — but the methodology is. Five focused bisections
below.

In [ ]:
from mamba_dvc.pipeline.correlate import correlate
from mamba_dvc.pipeline._internal import NCCMode, NCCNormalization

grid_positions = results[PairingStrategy.SEQUENTIAL].grid.positions
gt_small = series.field_at(grid_positions, 1.0) - series.field_at(grid_positions, 0.0)
gt_big = series.field_at(grid_positions, 15.0) - series.field_at(grid_positions, 0.0)
print(f'GT small (t=0->1)  mean ||u||: {float(np.linalg.norm(gt_small, axis=1).mean()):.4f} vx')
print(f'GT big   (t=0->15) mean ||u||: {float(np.linalg.norm(gt_big, axis=1).mean()):.4f} vx')
print(f'GT small per-axis mean: {gt_small.mean(axis=0)}')
print(f'GT big   per-axis mean: {gt_big.mean(axis=0)}')

def stat_row(label, field, gt):
    err = field.displacements - gt
    valid = field.valid
    if not bool(valid.any()):
        return f'  {label:38s}  no valid POIs'
    mag = np.linalg.norm(err[valid], axis=1)
    rec_mean = field.displacements[valid].mean(axis=0)
    return (
        f'  {label:38s}  MAE={float(mag.mean()):7.4f}  p95={float(np.percentile(mag,95)):7.4f}  '
        f'<u_rec>=({rec_mean[0]:+.3f},{rec_mean[1]:+.3f},{rec_mean[2]:+.3f})  '
        f'n_valid={int(valid.sum())}/{field.positions.shape[0]}'
    )

### Step 1 — Multi-GPU dispatcher vs single-GPU in-process

`device_ids=[0]` skips multiprocessing entirely; same algorithm,
different process model. If results agree, the bug is algorithmic, not in
the multi-process pool.

In [ ]:
print('=== Small displacement: pair (frames[0], frames[1]) ===')
print(stat_row('multi-GPU correlate_series[0]',
               results[PairingStrategy.SEQUENTIAL].fields[0], gt_small))
with MultiGPUDispatcher(device_ids=[0], volume_shape=VOLUME_SHAPE,
                        window=WINDOW, overlap=OVERLAP) as d1:
    f_small_d1 = d1.correlate(series.frames[0], series.frames[1])
    f_big_d1 = d1.correlate(series.frames[0], series.frames[15])
print(stat_row('single-GPU dispatcher [0]', f_small_d1, gt_small))

print()
print('=== Large displacement: pair (frames[0], frames[15]) ===')
print(stat_row('multi-GPU correlate_series[-1]',
               results[PairingStrategy.REFERENCE_ANCHORED].fields[-1], gt_big))
print(stat_row('single-GPU dispatcher [0]', f_big_d1, gt_big))

### Step 2 — Integer-shift sanity via `np.roll`

If even a hand-built `np.roll((2,-3,1))` pair fails, the bug is below the
warp (search-radius gate, outlier test, or sign convention).

In [ ]:
int_shift = (2, -3, 1)
ref_int = series.frames[0]
def_int = np.roll(ref_int, shift=int_shift, axis=(0, 1, 2))

with MultiGPUDispatcher(device_ids=[0], volume_shape=VOLUME_SHAPE,
                        window=WINDOW, overlap=OVERLAP) as d1:
    f_int = d1.correlate(ref_int, def_int)

gt_int = np.zeros_like(grid_positions)
gt_int[:, 0] = int_shift[0]
gt_int[:, 1] = int_shift[1]
gt_int[:, 2] = int_shift[2]
print(f'=== integer shift {int_shift} via np.roll ===')
print(stat_row('single-GPU dispatcher [0]', f_int, gt_int))

### Step 3 — Outlier status histograms across pairs

If many POIs at small `t` are flagged `OUTLIER` while large `t` is clean,
the host-side outlier test is misfiring when the median displacement is
near zero (dispersion estimate dominated by noise).

In [ ]:
from mamba_dvc.types import POIStatus

def status_row(label, fields):
    print(f'  {label}:')
    print(f'    pair   ' + '  '.join(f'{s.name:>10s}' for s in POIStatus))
    for i, f in enumerate(fields):
        counts = [int(np.count_nonzero(f.status == s)) for s in POIStatus]
        print(f'    t={i+1:2d}   ' + '  '.join(f'{c:10d}' for c in counts))

status_row('REFERENCE_ANCHORED', results[PairingStrategy.REFERENCE_ANCHORED].fields)
print()
status_row('SEQUENTIAL',         results[PairingStrategy.SEQUENTIAL].fields)

### Step 4 — Search-radius sweep at small displacement

Default `SEARCH_RADIUS = window//2 = 32` admits peaks up to 32 vx off zero.
If the texture has secondary autocorrelation lobes within ~6 vx of zero,
the peakfit can latch onto a sibling at small true displacement. Tightening
the basin to ±4 vx should collapse the error if peak ambiguity is the
cause.

In [ ]:
radii = [32, 8, 4, 2]
print('=== pair (frames[0], frames[1])  small |u| ~ 0.97 vx ===')
for r in radii:
    with MultiGPUDispatcher(device_ids=[0], volume_shape=VOLUME_SHAPE,
                            window=WINDOW, overlap=OVERLAP,
                            search_radius=r) as d1:
        f = d1.correlate(series.frames[0], series.frames[1])
    print(stat_row(f'search_radius={r:2d}', f, gt_small))

### Step 5 — NCC kernel A/B at small displacement

Re-runs the same A/B from §8 of `dvc-debug-texture.ipynb` on the failing
small-`t` pair: `linear`+`overlap`+`alpha=0` (current default) vs
`cyclic`+`global`+`alpha=0.25` (legacy). If linear-overlap is the worse one
on this case, the fix that landed for static synthetic pairs is somehow
miscalibrated on the (frames[0], frames[1]) data path.

In [ ]:
modes = [
    ('linear-overlap', NCCMode.LINEAR,  NCCNormalization.OVERLAP, None),
    ('cyclic-global',  NCCMode.CYCLIC,  NCCNormalization.GLOBAL,  None),
]
print('=== pair (frames[0], frames[1])  small |u| ===')
for label, mode, norm, alpha in modes:
    with MultiGPUDispatcher(device_ids=[0], volume_shape=VOLUME_SHAPE,
                            window=WINDOW, overlap=OVERLAP,
                            ncc_mode=mode, ncc_normalization=norm,
                            tukey_alpha=alpha) as d1:
        f = d1.correlate(series.frames[0], series.frames[1])
    print(stat_row(label, f, gt_small))

print()
print('=== pair (frames[0], frames[15])  large |u| ===')
for label, mode, norm, alpha in modes:
    with MultiGPUDispatcher(device_ids=[0], volume_shape=VOLUME_SHAPE,
                            window=WINDOW, overlap=OVERLAP,
                            ncc_mode=mode, ncc_normalization=norm,
                            tukey_alpha=alpha) as d1:
        f = d1.correlate(series.frames[0], series.frames[15])
    print(stat_row(label, f, gt_big))

### Step 6 — Verdict

Interpretation appended after execution, based on which probe lights up.